# RME Process Chatbot — live end-to-end demo

The whole backend, run start to finish on this machine, ending in a prompt you can type into —
in English or in Arabic.

```
 ../processes_pdf/*.pdf
        |  §1  extract      PyMuPDF native text, OCR fallback  ->  OCR-health audit only
        |
        |  §2  chunk        layout-based boundaries + MiniLM section labels
        v                   (reads the PDFs directly - layout isn't in extracted text)
 114 chunks + metadata + audit
        |  §3  index        BM25 (sparse) + all-MiniLM-L6-v2 -> FAISS (dense)
        v
 hybrid retriever
        |  §4  whitelist    every form number that really occurs in the corpus
        v
 §5  ask(question)
        |
        |     Arabic question?  -> translate AR->EN        (§5.5)
        v
      route -> retrieve top-3 -> qwen3:14b -> strip reasoning -> validate -> cite
                                     ^
                                     |  Arabic question? the answer is *generated*
                                     |  in Arabic here - not translated back
        v
      answer + sources + latency breakdown
```

**The model is `qwen3:14b`** and it is the only model this notebook runs. It was chosen on the
36-question eval whose results are in `model_eval_results.json`: 100% overall, 100% on the
`unanswerable` category, zero flagged citations. §5.5 explains what that choice does and does
not settle.

Arabic questions take the same path as English ones — retrieval, routing and the form-number
validator all operate on English throughout, and only the question is translated. The answer is
written in Arabic directly from the English context. Questions in Egyptian dialect are fine;
answers come back in formal Arabic. Details in §5.5.

Chunking is `adaptive_chunker.py`, which detects section boundaries from page layout rather than
from a list of expected heading strings.

§1 writes to `demo_extracted_raw.json` rather than the shipped `extracted_raw.json`, so no
artifact is overwritten; §2 re-chunks the PDFs and checks the result against the shipped
`chunks.json`.

**Run All**, then go to **§5.2** and put your own question in.

## 0. Setup

In [1]:
import json, sys, time, re, platform, textwrap
from pathlib import Path
from collections import Counter

# A Windows console is cp1252 and cannot encode Arabic at all. Jupyter is UTF-8
# already; this only matters when the notebook is executed headlessly through
# such a console (nbconvert from cmd.exe, say).
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

def _find_files_dir():
    """Locate the 'files' folder by marker file, not by assuming the CWD."""
    here = Path.cwd()
    for c in [here, here / "files", *here.parents]:
        if (c / "eval_set.json").exists() and (c / "retriever.py").exists():
            return c.resolve()
    raise SystemExit(f"Could not locate the 'files' folder from {here}")

FILES = _find_files_dir()
PDF_DIR = FILES.parent / "processes_pdf"
sys.path.insert(0, str(FILES))

import requests

OLLAMA_HOST = "http://localhost:11434"
MODEL = "qwen3:14b"     # the chosen model; see model_eval_results.json

print("files/       :", FILES)
print("processes_pdf:", PDF_DIR, "|", len(list(PDF_DIR.glob("*.pdf"))), "PDFs")
print("python       :", platform.python_version(), "|", platform.platform())

def ollama_models():
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
    r.raise_for_status()
    return [m["name"] for m in r.json().get("models", [])]

# ollama_placement() lives in chatbot.py and is imported in 5. It is not
# defined here because it reports on a *loaded* model, and nothing is
# loaded until 5.1 asks the first question.

try:
    have = ollama_models()
    ver = requests.get(f"{OLLAMA_HOST}/api/version", timeout=5).json()["version"]
    print(f"ollama {ver}  : up")
    print(f"  {MODEL:<12} {'pulled' if MODEL in have else 'MISSING -> ollama pull ' + MODEL}")
except Exception as e:
    print(f"ollama       : DOWN ({type(e).__name__}). Start `ollama serve` and re-run "
          "this cell. Sections 1-4 work without it; section 5 does not.")

files/       : D:\Model\RME-Chatbot-handoff\RME Chatbot\files
processes_pdf: D:\Model\RME-Chatbot-handoff\RME Chatbot\processes_pdf | 10 PDFs
python       : 3.12.9 | Windows-11-10.0.26200-SP0


ollama 0.32.6  : up
  qwen3:14b    pulled


## 1. Extract — PyMuPDF primary, OCR fallback

Real PDF bytes off disk. OCR fires only when a page's native text looks broken (under 20
characters, or an alphabetic ratio below 0.4) — what a genuinely scanned page looks like.

**This stage is no longer upstream of chunking.** §2 reads the PDFs directly, because the font
and layout signals it detects on exist only in the PDF and not in extracted text. What §1 is
now is the **OCR-health audit**, and that job matters more rather than less: §2 has no OCR path,
so any page without embedded text is a page the chunker cannot see. This cell is what tells you
which pages those are.

**The 9 `ocr_failed` lines below are expected, not a broken run.** Page 1 of every document is
an image-only signature/approval cover page. With no `tesseract` binary installed the fallback
raises, the page is logged loudly, and extraction continues on native text. No process content
lives on those cover pages — they carry the title, doc code and approval signatures, all of
which also appear in the document body. §2's `pages with no text` counter reports the same 9
pages from the other side.

In [2]:
import extract_pipeline as EX

DEMO_RAW = FILES / "demo_extracted_raw.json"
DEMO_LOG = FILES / "demo_ocr_fallback_log.json"

t0 = time.perf_counter()
extracted = EX.run(PDF_DIR, DEMO_RAW, DEMO_LOG, allow_ocr=True)
print(f"\nextraction wall time: {time.perf_counter() - t0:.1f}s")

CS_Signature_Matrix.pdf: 3 pages, 0 native, 3 FELL BACK
PCM01_Customer_Satisfaction_Process_1.pdf: 6 pages, 5 native, 1 FELL BACK
PCM02_Branding_for_Construction_Sites_Process_1.pdf: 4 pages, 3 native, 1 FELL BACK
PCN01_Subcontract_Agreement_Process_1.pdf: 11 pages, 10 native, 1 FELL BACK


PQD12_Quality_Plan_Inspection_and_Testing_Process.pdf: 8 pages, 7 native, 1 FELL BACK
PSE01_RME_SelfExecution_Process_1.pdf: 5 pages, 4 native, 1 FELL BACK
PTN01_Project_Initiation_Process.pdf: 6 pages, 5 native, 1 FELL BACK
PTN02_Project_Launching_Process.pdf: 7 pages, 6 native, 1 FELL BACK
PVMO01_Vendor_selection_and_Bidding_Process.pdf: 8 pages, 7 native, 1 FELL BACK
PVMO02_Procurement_Process.pdf: 8 pages, 7 native, 1 FELL BACK



10 docs, 66 pages in 0.2s
OCR fallback pages: 12/66 (18.2%)
Saved  D:\Model\RME-Chatbot-handoff\RME Chatbot\files\demo_extracted_raw.json
Log    D:\Model\RME-Chatbot-handoff\RME Chatbot\files\demo_ocr_fallback_log.json

extraction wall time: 0.2s


## 2. Chunk — structural boundaries, semantic labels

Fixed-size windows would cut a process step away from its own form number, which is exactly
what most questions ask for. So chunking follows the document's own sections. `adaptive_chunker.py`
does that in two independent stages:

- **Where a section starts** is decided from *layout* — PyMuPDF gives font size and bold flags
  per span, and a heading is a short line that is visually distinct from body text. Where layout
  is flat (several of these documents style headings identically to body text) it falls back to
  the numbering pattern `1. SOMETHING`. Neither signal reads the heading's wording.
- **What the section is** is decided by comparing the heading to a canonical taxonomy: exact
  match, then fuzzy match, then MiniLM cosine — reusing the embedding model the retriever
  already loads. That is what absorbs `STAKHOLDER`, `PROCES INPUT` and `Process Operations`
  without a keyword list naming each one.

This replaced an earlier chunker that matched literal strings from a hand-written
`SECTION_HEADERS` list and therefore did both jobs at once. It worked on these 9 documents and
failed silently on anything else — a document saying `RESPONSIBLE PARTIES` instead of
`STAKEHOLDER` matched nothing, produced one document-sized chunk, and reported success.

Every chunk still carries `doc_code`, `title`, `filename`, `section` and `step`, plus the raw
heading and which tier labeled it, so an answer can cite where it came from and the chunking
itself is auditable.

**Two things this stage does not do.** It has no OCR path — it reads PDF layout, which only
exists for native text, so the 9 image-only cover pages from §1 are simply not chunked (the
`pages with no text` counter below is that fact, stated out loud). And `step` is `None` on every
chunk: step-level splitting inside `PROCESS OPERATION` is not implemented. Neither matters at 9
documents; the second is the one that bites at 500–600, where a section grows long enough that a
step and its form number land in different chunks.

In [3]:
import adaptive_chunker as AC
from retriever import route_prefixes          # doc-code routing lives with retrieval

t0 = time.perf_counter()
chunks, chunk_report = AC.chunk_corpus(PDF_DIR)
print(f"{len(chunks)} chunks from {chunk_report['docs']} documents "
      f"in {time.perf_counter() - t0:.1f}s")

# Chunking the PDFs here should reproduce the shipped artifact exactly.
shipped = json.load(open(FILES / "chunks.json", encoding="utf-8"))
identical = [c["text"] for c in chunks] == [c["text"] for c in shipped]
print(f"identical to shipped chunks.json ({len(shipped)} chunks): {identical}")

detectors = Counter()
for r in chunk_report["per_doc"]:
    detectors.update(r["detection"]["detectors"])

print(f"\nboundary detectors : {dict(detectors)}")
print(f"label methods      : {chunk_report['label_methods']}")
print(f"windowed fallback  : {chunk_report['docs_windowed_fallback']}/{chunk_report['docs']} docs")
print(f"unmatched headings : {len(chunk_report['unmatched_headings'])}")
print(f"pages with no text : {chunk_report['pages_with_no_native_text']}"
      "  (the image-only cover pages from §1 - no OCR path, so not chunked)")
print(f"skipped as scanned : {chunk_report['skipped_scanned'] or 'none'}"
      "  (fully-scanned PDFs -> §2.5, workflow_extractor.py)")

print("\nchunks per document:")
for fn, n in Counter(c["filename"] for c in chunks).most_common():
    print(f"  {n:>3}  {fn}")

print("\nsections found:", ", ".join(sorted({c["section"] for c in chunks})))

ex = next((c for c in chunks if "PROCESS OPERATION" in c["section"]), chunks[0])
print(f"\nexample chunk -- {ex['filename']}")
print(f"  section={ex['section']!r}  from heading {ex['raw_heading']!r}")
print(f"  labeled by {ex['label_method']} ({ex['label_score']})  doc_code={ex['doc_code']}")
print("  " + textwrap.shorten(ex["text"], 300))

D:\Model\RME-Chatbot-handoff\RME Chatbot\venv_chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7356.05it/s]

114 chunks from 9 documents in 9.8s
identical to shipped chunks.json (114 chunks): True

boundary detectors : {'numbering': 103, 'lone_number+stitch': 8, 'font_size': 3}
label methods      : {'front_matter': 9, 'exact': 80, 'fuzzy': 25}
windowed fallback  : 0/9 docs
unmatched headings : 0
pages with no text : 9  (the image-only cover pages from §1 - no OCR path, so not chunked)
skipped as scanned : ['CS Signature Matrix.pdf']  (fully-scanned PDFs -> §2.5, workflow_extractor.py)

chunks per document:
   13  PCM01_Customer_Satisfaction_Process_1.pdf
   13  PCN01_Subcontract_Agreement_Process_1.pdf
   13  PQD12_Quality_Plan_Inspection_and_Testing_Process.pdf
   13  PSE01_RME_SelfExecution_Process_1.pdf
   13  PTN02_Project_Launching_Process.pdf
   13  PVMO01_Vendor_selection_and_Bidding_Process.pdf
   12  PCM02_Branding_for_Construction_Sites_Process_1.pdf
   12  PTN01_Project_Initiation_Process.pdf
   12  PVMO02_Procurement_Process.pdf

sections found: DOCUMENT CHANGE HISTORY, FRONT MATT

## 2.5 The other kind of document — scanned workflow diagrams

Not every PDF in this corpus is a process document. `CS Signature Matrix.pdf` is three pages of
**pure image, zero extractable text** — approval flowcharts with roles in boxes, decision
diamonds, rejection loops, and red monetary thresholds deciding who signs.

`adaptive_chunker.py` cannot see it at all, and before `workflow_extractor.py` existed it failed
*silently*: dropping the file into `processes_pdf/` moved the `pages with no text` counter from
9 to 12 and produced nothing else.

OCR alone does not rescue it either, and that is the point. The meaning is in the **edges** —
which arrow points where, and which threshold labels it. OCR returns a bag of role names with
every relationship destroyed. So the routing decision is made on extractable text:

```
  every page has < 20 characters?  -> workflow: a vision model reads the topology
  otherwise                        -> process: the normal layout-based chunker
```

"Every page", not "any page": each process document has one image-only signature cover page,
so "any page needs OCR" would misfile all nine.

`qwen3.6:27b` (`qwen3:14b` has no vision capability) emits a **structured graph** — nodes,
edges, conditions, per lane — and the prose that gets embedded is rendered deterministically
from that graph. Free prose would read better and be unauditable; a paraphrased threshold is
invisible, and thresholds are what people ask about.

**One chunk per lane, not per file.** This single PDF holds six distinct approval flows. As one
chunk, a question about steel would retrieve caravans and formwork too and leave the model to
guess which chain applies.

**These chunks are the first part of the corpus that is not ground truth**, and they are treated
accordingly: they carry `source_type: "vlm_description"` and `review: "machine"`, they are
excluded from the validator's whitelist in §4, and the app labels any answer built from them.
Three runs at temperature 0 misread the same crowded junction three different ways — dropping a
threshold, inventing a reverse arrow to hang it on, and duplicating an arrow to hang it on. The
extractor's audit catches all three shapes; it cannot catch a clean misreading, which is what
`review: "machine"` is for.

## 3. Index — hybrid BM25 + dense, with doc-code routing

Two channels, min-max normalised and mixed at `dense_weight=0.4`:

- **BM25** is the exact-match channel. Form numbers are literal strings; a dense model has no
  reason to keep `F-P-CN-01-11` and `F-P-VMO-01-01` apart. It indexes **title + section + body**,
  not the body alone, and applies a light plural strip. Both were added after a real miss: every
  *plural* stakeholder question returned `OBJECTIVES` while the singular form answered correctly.
  Two causes compounded — PVMO01's heading is misspelled in the PDF as `2. STAKHOLDER`, so the
  body held no matching token (the fuzzy labeller had recovered the correct name into `section`,
  which BM25 could not see), and with no stemming `stakeholders` ≠ `stakeholder`. Indexing the
  canonical label fixes that for any short, list-shaped section identified mainly by its heading.
- **MiniLM → FAISS `IndexFlatIP`** is the paraphrase channel ("who signs off on" ≈ "approved by").

Before either runs, `route_prefixes()` narrows the candidate pool to a single process family
when the query unambiguously names one. On the eval set that fires on 22 of 36 questions and
never once excluded the gold document.

Nothing downloads here: MiniLM loads from the local HF cache and chunk embeddings are cached in
`.embed_cache/` keyed by a corpus+model fingerprint. The slow part below is importing torch,
not encoding.

Note for §5.5: every one of these channels is English-only — BM25 tokenises `[a-z0-9]+`, MiniLM
was trained on English, and `PREFIX_HINTS` is an English keyword table. That is the reason
Arabic is handled by translating the query rather than by touching anything here.

In [4]:
from retriever import Retriever
from chatbot import load_corpus, ground_truth_chunks
import validator as V

# The corpus is not just the process text any more. Scanned workflow diagrams
# are read by a vision model into workflow_chunks.json (workflow_extractor.py)
# and indexed alongside, because a user does not know which kind of document
# answers their question.
CORPUS = load_corpus()
n_vlm = len(CORPUS) - len(ground_truth_chunks(CORPUS))
print(f"corpus: {len(CORPUS)} chunks = {len(CORPUS) - n_vlm} from process text "
      f"+ {n_vlm} from scanned workflow diagrams")

t0 = time.perf_counter()
R = Retriever(CORPUS)      # weighted fusion, dense_weight=0.4, routing on
print(f"index built in {time.perf_counter() - t0:.1f}s")
print(f"model={R.model_name}  embeddings={R.embeddings.shape}  "
      f"fusion={R.fusion} (dense_weight={R.dense_weight})")

probe = "What form is used for the Customer Satisfaction Survey?"
print(f"\nroute for {probe!r} -> {route_prefixes(probe) or 'no route, search everything'}")
for h in R.search(probe, top_k=3):
    print(f"  [{h['score']:.3f}] {h['filename'][:44]:<44} {h['section'][:26]}")

corpus: 120 chunks = 114 from process text + 6 from scanned workflow diagrams


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4712.60it/s]

index built in 0.2s


model=all-MiniLM-L6-v2  embeddings=(120, 384)  fusion=weighted (dense_weight=0.4)

route for 'What form is used for the Customer Satisfaction Survey?' -> ['PCM']
  [0.937] PCM01_Customer_Satisfaction_Process_1.pdf    PROCESS OPERATION
  [0.888] PCM01_Customer_Satisfaction_Process_1.pdf    PROCESS INPUT
  [0.885] PCM01_Customer_Satisfaction_Process_1.pdf    PERFORMANCE MEASURES


## 4. Validator — deterministic hallucination proxy

No model judges another model. Pull every form-shaped string out of an answer with a
deliberately *loose* regex (so a malformed invention like `F-P-CM-1-1` is caught too), then
test each against the set of form numbers that actually occur in the corpus.

It **flags, it does not block.** At 9 documents the whitelist is a subset of reality — the
corpus references form families (FW, HR, OP, PU, QP) whose source documents aren't here yet —
so "unknown" today means *unverifiable*, not *fabricated*. `policy="block"` becomes safe once
the full 500–600 document corpus is indexed.

In [5]:
# Verbatim source text ONLY. Letting a vision model's reading into the
# whitelist would let a form number it invented whitelist itself - the
# validator would then confirm it as genuine, which is worse than no validator.
TRUTH = ground_truth_chunks(CORPUS)
FORM_WHITELIST = V.build_form_whitelist(TRUTH)
DOC_WHITELIST = V.build_doc_code_whitelist(TRUTH)
print(f"{len(FORM_WHITELIST)} known form numbers, {len(DOC_WHITELIST)} known doc codes "
      f"(from {len(TRUTH)} verbatim chunks; {len(CORPUS) - len(TRUTH)} VLM chunks excluded)")

for p in [
    "The Customer Satisfaction Survey uses form F-P-CM-01-01.",   # real
    "Use form F-P-CM-03-07 to request a company car.",            # invented
    "Not specified in these process documents.",                  # no citation
]:
    v = V.validate_answer(p, FORM_WHITELIST, DOC_WHITELIST)
    print(f"  {v['verdict']:<8} cited={v['cited']} unknown={v['unknown']}   <- {p[:52]}")

67 known form numbers, 9 known doc codes (from 114 verbatim chunks; 6 VLM chunks excluded)
  clean    cited=['F-P-CM-01-01'] unknown=[]   <- The Customer Satisfaction Survey uses form F-P-CM-01
  flagged  cited=['F-P-CM-03-07'] unknown=['F-P-CM-03-07']   <- Use form F-P-CM-03-07 to request a company car.
  clean    cited=[] unknown=[]   <- Not specified in these process documents.


## 5. The chatbot

`ask()` is the whole pipeline in one call: detect the language → translate to English if the
question is Arabic → route → retrieve top-3 → build a context-only prompt → generate → strip
any reasoning trace → validate the citations → print it with its sources and a latency
breakdown.

Generation is deterministic (`temperature=0`, `seed=0`), so a repeated question gives a
repeated answer. The prompt forbids answering from parametric memory and requires a filename
citation — that instruction is what makes the refusal in §5.4 work.

**Answers run to completion — `num_predict: -1`.** This used to be capped at 200 tokens, which
was invisible on every question in this notebook and in the eval set, because a form number, a
deadline or a refusal all finish well under it. An explanatory question does not: *"explain the
process operations in procurement"* hit the cap and stopped mid-sentence, with nothing in the
output saying so. The cap is a *maximum*, not a target — short answers still stop on their own,
so removing it costs nothing on the questions above and only spends time when an answer
genuinely needs the room.

What bounds generation now is `num_ctx`, set explicitly to 8192. Leaving it unset would take
Ollama's default (4096) and quietly reintroduce a ceiling that shrinks as `top_k` or chunk size
grows. The remaining trade is honest: with no output cap, a vague question has no ceiling at
all — tens of seconds on the GPU this run used, and minutes on CPU, with only the 1800s HTTP
timeout as a backstop.

`strip_reasoning()` earns its place: `qwen3:14b` is a reasoning model. Requests set
`think: False` to suppress traces at the source, and anything that still slips through is cut
before validation — otherwise a model *musing* "it might be F-P-CM-01-01" would register as a
confident citation it never actually made. The same wrapper is handed to `translate.py`, so a
trace can't leak into a translation either.

**On an Arabic question the answer is generated in formal Arabic, so validation runs on Arabic
text.**
That works because `FORM_RE` in `validator.py` is Latin-script and the prompt pins form numbers,
doc codes and filenames to Latin script with Western digits. `ask()` does not take that on
trust — it flags any answer containing Arabic-Indic digits, since a code written in those is
invisible to the validator rather than merely wrong. §5.5 has the rest of the Arabic design.

In [6]:
# The pipeline itself lives in chatbot.py, not in this cell.
#
# It moved there when app.py (the Streamlit front end) arrived. The alternative
# was two copies of the prompt text, the reasoning-strip regex and the
# validation order - and those are exactly the things that drift apart
# silently. The prose in this section describes what chatbot.py does; read them
# together.
#
# The seam: Chatbot.ask() returns a record and prints nothing. Rendering is the
# caller's job, which is why show() below lives here and app.py has its own.
# Everything upstream of the return value is shared.
from chatbot import Chatbot, ollama_placement

# Reuse the retriever built in §3 rather than letting Chatbot build a second
# one - that would reload MiniLM and re-encode the corpus for no reason.
BOT = Chatbot(CORPUS, model=MODEL, retriever=R)
print(f"pipeline ready | {len(BOT.form_whitelist)} forms, {len(BOT.doc_whitelist)} doc codes "
      f"| model={BOT.model} | validator policy={BOT.policy!r}")


def show(rec, top_k=3, show_context=False):
    """Print a record: the answer, then the sources and verdict it has to be
    read against.

    That pairing is the whole point of the layout. An answer on its own cannot
    tell you whether a wrong result was a retrieval failure or a generation
    failure, and those have completely different fixes.
    """
    v, src_ar = rec["validator"], rec["lang"] == "ar"
    print("=" * 78)
    print(f"Q: {rec['question']}")
    print("=" * 78)
    print(textwrap.fill(rec["display"], 78, initial_indent="  ", subsequent_indent="  "))
    print(f"\n  sources (top-{top_k}, route={rec['route'] or 'none'}):")
    for h in rec["hits"]:
        step = f"  |  step {h['step']}" if h["step"] else ""
        print(f"    [{h['score']:.3f}] {h['filename']}  |  {h['section']}{step}")
    flag = "clean" if v["ok"] else f"FLAGGED - unverifiable form(s): {', '.join(v['unknown'])}"
    print(f"\n  cited forms : {v['cited'] or 'none'}   validator: {flag}")

    lat = f"retrieval {rec['retrieval_s']*1000:.0f} ms + generation {rec['generation_s']:.1f} s"
    if src_ar:
        lat = f"translate-in {rec['translate_in_s']:.1f} s + " + lat
        if rec["translate_out_s"] > 0.05:
            lat += f" + refusal-translate {rec['translate_out_s']:.1f} s"
    print(f"  latency     : {lat} = {rec['total_s']:.1f} s   ({rec['model']})")
    if rec["had_reasoning"]:
        print("  note        : a reasoning trace was stripped before validation")
    if rec["arabic_digits"]:
        print(f"  WARNING     : answer contains Arabic-Indic digits "
              f"({''.join(rec['arabic_digits'])}). Any code written in them is "
              "unlookuppable and invisible to the validator.")
    if show_context:
        if src_ar:
            print(f"\n  --- question as translated for retrieval ---\n    {rec['question_en']}")
        print("\n  --- context sent to the model ---")
        for h in rec["hits"]:
            print(f"\n  [{h['filename']} | {h['section']}]")
            print(textwrap.indent(textwrap.fill(h["text"], 74), "    "))
    print()


def ask(question, top_k=3, show_context=False, quiet=False):
    """Ask and render. Returns the record either way."""
    rec = BOT.ask(question, top_k=top_k)
    if not quiet:
        show(rec, top_k=top_k, show_context=show_context)
    return rec

pipeline ready | 67 forms, 9 doc codes | model=qwen3:14b | validator policy='flag'


### 5.1 Worked examples

Three shapes the corpus supports — a form lookup, a numeric deadline, and a role lookup. The
first call also pays the model's cold-load cost (`qwen3:14b` is ~9 GB off disk), so its latency
is the outlier; re-run the cell and it drops.

**Read every latency here against the hardware line printed at the bottom of the cell.** This
run has the model resident in GPU memory, which is why short answers land in ~3 s. The same
questions on a CPU-only machine run 15–70 s — a 5–20× spread that no amount of prompt or
retrieval work would close. The retrieval numbers (~10–40 ms) are CPU-bound either way.

Every answer prints with the chunks it was built from and the validator's verdict. That pairing
is the whole point of the layout: an answer on its own cannot tell you whether a wrong result
was a retrieval failure or a generation failure, and those have completely different fixes. A
refusal in particular looks like the system being careful — the only way to know is to read the
sources printed underneath it.

In [7]:
_ = ask("What form is used for the Customer Satisfaction Survey?")
_ = ask("Within how many hours must the PM send corrective action plans "
        "after a customer satisfaction gap is reported?")
_ = ask("Who is responsible for issuing an NCR when there is a gap between "
        "customer perception and RME's expected standard?")

print(f"hardware    : {ollama_placement()}")

Q: What form is used for the Customer Satisfaction Survey?
  The form used for the Customer Satisfaction Survey is F-P-CM-01-01. (Doc
  filename: PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.937] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.888] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS INPUT
    [0.885] PCM01_Customer_Satisfaction_Process_1.pdf  |  PERFORMANCE MEASURES

  cited forms : ['F-P-CM-01-01']   validator: clean
  latency     : retrieval 10 ms + generation 3.1 s = 3.1 s   (qwen3:14b)



Q: Within how many hours must the PM send corrective action plans after a customer satisfaction gap is reported?
  The PM must send corrective action plans within 24 hours after a customer
  satisfaction gap is reported. (Doc:
  PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [1.000] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.680] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS CONTROL
    [0.675] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OUTPUT

  cited forms : none   validator: clean
  latency     : retrieval 12 ms + generation 2.9 s = 2.9 s   (qwen3:14b)



Q: Who is responsible for issuing an NCR when there is a gap between customer perception and RME's expected standard?
  QA is responsible for issuing an NCR when there is a gap between customer
  perception and RME's expected standard. (Doc filename:
  PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.923] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.728] PCM01_Customer_Satisfaction_Process_1.pdf  |  OBJECTIVES
    [0.616] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS CONTROL

  cited forms : none   validator: clean
  latency     : retrieval 13 ms + generation 3.0 s = 3.0 s   (qwen3:14b)



hardware    : qwen3:14b: fully on GPU, 10.5 GB


### 5.2 Ask your own question — edit this cell and re-run it

Put anything in `MY_QUESTION` and run the cell (`Ctrl`+`Enter`). Nothing above needs re-running;
the index stays in memory.

**Arabic works here too** — `ask()` detects the script and routes the question through §5.5's
translation path on its own, no flag to set. Egyptian dialect is fine; so is code-switching
("الـ procurement process بيبدأ ازاي"), because the language check counts the fraction of
*letters* in Arabic script rather than requiring a majority.

`show_context=True` prints the exact chunks the model was handed, which is how you tell a
retrieval failure from a generation failure when an answer looks wrong. On an Arabic question it
also prints the English the query was translated into — the other thing worth checking, since a
mistranslated domain term is a retrieval miss that nothing else in the output will reveal.

The corpus answers form numbers, step responsibilities, deadlines in days and hours,
definitions, and who approves what — across customer satisfaction, site branding, subcontract
agreements, quality inspection and testing, self-execution, project initiation, project
launching, vendor selection and procurement.

**Broad "explain the whole process" questions work, and they cost proportionally more.** Answers
are no longer length-capped (§5), so *"Explain to me the process operations in procurement"*
returns all 18 steps — about 1,240 tokens. On the GPU this run used that is tens of seconds; on
a CPU-only machine the same answer took roughly 8 minutes. That is the honest cost of not being
cut off mid-sentence, and it scales with output length rather than being a fixed penalty. The
short lookups above are ~3 s here once the model is warm, 20–30 s on CPU.

In [8]:
MY_QUESTION = "What is the form number for risk assessment?"

_ = ask(MY_QUESTION, show_context=False)

Q: What is the form number for risk assessment?
  The form number for risk assessment is F-P-QD-05-01. This information is
  specified in the documents
  PCM02_Branding_for_Construction_Sites_Process_1.pdf,
  PTN02_Project_Launching_Process.pdf, and
  PTN01_Project_Initiation_Process.pdf.

  sources (top-3, route=none):
    [0.961] PCM02_Branding_for_Construction_Sites_Process_1.pdf  |  PROCESS RISK ASSESSMENT
    [0.945] PTN02_Project_Launching_Process.pdf  |  PROCESS RISK ASSESSMENT
    [0.942] PTN01_Project_Initiation_Process.pdf  |  PROCESS RISK ASSESSMENT

  cited forms : ['F-P-QD-05-01']   validator: clean
  latency     : retrieval 10 ms + generation 3.4 s = 3.4 s   (qwen3:14b)



### 5.3 Interactive prompt

A REPL, if you'd rather not edit a cell each time. Run the cell, type questions in either
language, and press Enter on a blank line (or type `quit`) to stop.

In [9]:
def chat():
    print(f"RME process chatbot | {MODEL} | ask in English or Arabic")
    print("blank line or 'quit' to exit\n")
    while True:
        try:
            q = input("you> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n(stopped)")
            return
        if not q or q.lower() in {"quit", "exit", "q"}:
            print("(stopped)")
            return
        try:
            ask(q)
        except Exception as e:
            # One bad question shouldn't drop you out of the session.
            print(f"  ! {type(e).__name__}: {e}\n")


try:
    chat()
except Exception as e:      # headless execution has no stdin
    print(f"interactive prompt unavailable here ({type(e).__name__}: {e}). "
          "Run this cell yourself in Jupyter or VS Code.")

RME process chatbot | qwen3:14b | ask in English or Arabic
blank line or 'quit' to exit

interactive prompt unavailable here (StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.). Run this cell yourself in Jupyter or VS Code.


### 5.4 The case that matters: a question the documents do not answer

The 9 documents say nothing about milestone penalties. A model answering from its own weights
would happily invent a clause and a form number to go with it. Retrieval still returns its three
nearest chunks — it always returns something — so the refusal has to come from the prompt, and
the validator independently confirms nothing was fabricated on the way out.

In [10]:
_ = ask("What is the penalty amount if a subcontractor misses a milestone date?")

Q: What is the penalty amount if a subcontractor misses a milestone date?
  Not specified in these process documents.

  sources (top-3, route=['PCN']):
    [0.791] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS OPERATION
    [0.724] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS RISK ASSESSMENT
    [0.635] PCN01_Subcontract_Agreement_Process_1.pdf  |  OBJECTIVES

  cited forms : none   validator: clean
  latency     : retrieval 11 ms + generation 3.2 s = 3.2 s   (qwen3:14b)



### 5.5 Arabic questions

RME's users write in Arabic, and mostly in Egyptian dialect. The corpus does not — and neither
do the chunk embeddings, the BM25 tokeniser (`[a-z0-9]+`), the routing table
`PREFIX_HINTS` in `retriever.py`, or the `F-P-…` regex in `validator.py`. Making all of that
multilingual is a rebuild of the entire retrieval stack, for nine documents.

So `translate.py` translates the **question** and leaves the middle alone:

```
  ايه هي الاستمارة اللي بتستخدم في استبيان رضا العملاء؟
      |  is_arabic()   fraction of *letters* written in Arabic script - the denominator
      |                is letters only, so a Latin form number inside the question
      v                cannot drag it under the threshold
  "What is the form used in the customer satisfaction survey?"
      |
      |  the existing pipeline, untouched: route -> retrieve -> generate -> validate
      v
  "النموذج المستخدم في استبيان رضا العملاء هو F-P-CM-01-01."
```

`qwen3:14b` does the translating as well as the answering, so this adds no dependency, needs no
extra download, and nothing leaves the machine.

**Asymmetric by design: dialect in, formal Arabic out.** Users write Egyptian dialect, so the
inbound translation has to cope with it. Answers go out in Modern Standard Arabic, which is the
right register for a reply quoting ISO process documents and also the one `qwen3:14b` produces
reliably.

**The answer is generated in Arabic, not translated into it.** The prompt carries an
`ANSWER_IN_ARABIC` directive and the model writes Arabic straight from the English context.
An earlier version wrote an English answer and ran it back through `to_arabic`; that second hop
is gone. It bought nothing and cost two things: 6–18 s per question on the machine that produced
the outputs committed above, and a second opportunity to corrupt a form number. The only thing
still translated on the way out is the `policy="block"` refusal, because that string comes from
`validator.py` rather than the model.

**Form numbers are masked, not merely requested.** `mask_codes()` swaps every `F-P-…` and `P-…`
out for a placeholder before the text reaches the model and puts it back afterwards, so a code
cannot be corrupted by a translation that never saw it. The verbatim-Latin instruction in the
prompt stays as a second line of defence for the identifiers the regex does not cover, mainly
PDF filenames. This is belt and braces on purpose: `validator.py` decides "hallucinated
citation" by regex, so one mangled digit turns a correct answer into a false flag, and on the
way in it silently breaks doc-code routing.

**Validation now runs on the Arabic answer.** That is safe because `FORM_RE` is Latin-script and
the codes are pinned to Latin script — but "safe because the prompt says so" is exactly the kind
of assumption that fails quietly, so `ask()` checks: if the answer contains Arabic-Indic digits
(`٠-٩`), it prints a warning, because a code written in those is both unlookuppable and
invisible to the validator. On the questions below the check has never fired.

**Two costs, stated plainly.** An Arabic question makes two generation calls instead of one, so
expect roughly 2× the latency of the equivalent English question. And the pipeline is only as
good as the translation: a mistranslated domain term becomes a retrieval miss with nothing in
the output to signal it. That is why the record keeps `question_en` and `show_context=True`
prints it.

**What is measured.** `eval_arabic.py` scores the Arabic path on **retrieval hit** — the one
metric whose ground truth is language-independent, since the gold document per question is the
same whatever language it was asked in. All 36 eval questions, same gold documents as the
English run:

```
English (baseline)    32/32 = 100.0%
Arabic -> English     31/32 =  96.9%
Arabic (raw, no MT)   18/32 =  56.2%      <- control
```

The control row is the one that earns its place, and it did not land where expected. The
prediction was near-zero, on the reasoning that BM25 yields no tokens for Arabic and
`PREFIX_HINTS` never fires. 56% is not near zero — with BM25 flat, ranking falls entirely to
the dense channel, and MiniLM keeps enough cross-lingual signal to beat chance (which is
already ~33% here: top-3 of 9 documents). The right reading is "somewhat better than
guessing", and the 41-point gap is what shows the translation step is carrying the result.

**The single miss is worth reading, not rounding away.** PTN01-3 asks about the *Commercial
dept*; the Arabic says `قسم التجارة`, which comes back as "trade department". That loses the
word "commercial" — both a BM25 term and the `PREFIX_HINTS` routing keyword for PCM. A
translation failure, not a retrieval failure, and precisely the silent-miss mode warned about
above: nothing in the output flags that a domain term was dropped. A term glossary pinned into
the translation prompt is the obvious fix.

**Two reasons the 96.9% is an upper bound, not a user-facing claim.** The Arabic questions are
generated by the same model that translates them back, so the round trip flatters itself. And
they came out in MSA despite being asked for in Egyptian, so the set does not exercise dialect
*input* — which is the half that still matters, now that output is formal Arabic by choice. The
dialect input path does work: the hand-written Egyptian questions below all retrieve correctly.
It is simply not what these 36 questions measure.

**Still unmeasured:** Arabic *answer* quality. Scoring it means `must_include` strings in
Arabic, and those have to be written by a person.

In [11]:
# The same three shapes as §5.1 and §5.4, asked in Egyptian dialect rather than
# MSA - dialect is what users actually type, and it is the harder input.

# expect F-P-CM-01-01, in Latin script, validator clean
_ = ask("ايه هي الاستمارة اللي بتستخدم في استبيان رضا العملاء؟")

# expect 24 hours
_ = ask("خلال كام ساعة لازم مدير المشروع يبعت خطط الإجراءات التصحيحية "
        "بعد ما يتبلغ عن فجوة في رضا العملاء؟")

# unanswerable - the refusal must come back in Arabic, and the validator must
# still report no fabricated citation
_ = ask("قيمة الغرامة كام لو مقاول الباطن اتأخر عن تاريخ التسليم؟")

# code-switched, and quotes a form number that has to survive the round trip
# through translation byte-identical - this is what mask_codes() is for
_ = ask("الـ F-P-CM-01-01 ده بيتستخدم في ايه؟", show_context=False)

Q: ايه هي الاستمارة اللي بتستخدم في استبيان رضا العملاء؟
  النموذج المستخدم في استبيان رضا العملاء هو F-P-CM-01-01.
  (PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.932] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.862] PCM01_Customer_Satisfaction_Process_1.pdf  |  PERFORMANCE MEASURES
    [0.855] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS INPUT

  cited forms : ['F-P-CM-01-01']   validator: clean
  latency     : translate-in 2.7 s + retrieval 10 ms + generation 3.2 s = 5.9 s   (qwen3:14b)



Q: خلال كام ساعة لازم مدير المشروع يبعت خطط الإجراءات التصحيحية بعد ما يتبلغ عن فجوة في رضا العملاء؟
  24 ساعات   المصدر: PCM01_Customer_Satisfaction_Process_1.pdf

  sources (top-3, route=['PCM']):
    [1.000] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.689] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS CONTROL
    [0.655] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OUTPUT

  cited forms : none   validator: clean
  latency     : translate-in 2.8 s + retrieval 13 ms + generation 2.8 s = 5.6 s   (qwen3:14b)



Q: قيمة الغرامة كام لو مقاول الباطن اتأخر عن تاريخ التسليم؟
  غير محدد في وثائق العمليات هذه.

  sources (top-3, route=['PCN']):
    [0.983] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS RISK ASSESSMENT
    [0.826] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS OPERATION
    [0.781] PCN01_Subcontract_Agreement_Process_1.pdf  |  OBJECTIVES

  cited forms : none   validator: clean
  latency     : translate-in 2.8 s + retrieval 10 ms + generation 3.2 s = 6.0 s   (qwen3:14b)



Q: الـ F-P-CM-01-01 ده بيتستخدم في ايه؟
  F-P-CM-01-01 يُستخدم لاستبيان رضا العملاء، كما ورد في وثيقة
  "PCM01_Customer_Satisfaction_Process_1.pdf".

  sources (top-3, route=['PCM']):
    [0.930] PCM02_Branding_for_Construction_Sites_Process_1.pdf  |  RELATED DOCUMENTED INFORMATION
    [0.906] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.897] PCM01_Customer_Satisfaction_Process_1.pdf  |  RELATED DOCUMENTED INFORMATION

  cited forms : ['F-P-CM-01-01']   validator: clean
  latency     : translate-in 2.7 s + retrieval 11 ms + generation 3.2 s = 5.9 s   (qwen3:14b)



### 5.6 A question only the scanned diagram can answer

Nothing in the nine process documents says who signs a comparison sheet above 3M — that lives
only in `CS Signature Matrix.pdf`, which has no extractable text. These questions are answerable
*only* through the vision path.

The middle one is the reason lane-level chunking matters: page 2 holds two flows side by side,
and the steel lane routes to the COO at **any value** while the non-steel lane only escalates
**over 500 K**. One chunk per file would hand the model both and let it choose.

**Three things this section taught, all found by testing rather than reasoning:**

*Phrasing of a generated chunk is part of the pipeline.* "At what value does the COO approve?"
was refused on a chunk retrieved at score **1.000** whose text held the answer — the generator
did not read `Any Value: approved by COO` as an answer to "what value". Retrieval was never the
problem. `render_prose()` now phrases thresholds as sentences pairing the word *value* with the
threshold text.

*Order has to be computed, not implied.* Asked who signs before and after the VP, the model read
`COO -> VP -> CFO` and answered that the **CFO comes before the VP** — confidently inverted. The
order is now derived from the graph and printed as a numbered list. Two corrections came out of
that: the walk must **stop at forks** (a first version linearised page 1's branch, dropped the
CEO and claimed "VP signs last", which broke the 3M question), and it must stay **terse** (a
sentence per step doubled the chunk and caused fresh refusals).

*More context can make things worse.* Sibling lanes are near-duplicates, so top-3 returns three
variants of one flow and only one may contain the role asked about. The before/after question
answers correctly at `top_k=1` and `2` and refuses at `3`. Process documents behave the opposite
way — three chunks from one document reinforce each other. Same retriever, opposite dynamics,
worth remembering as more scanned files arrive. A prompt directive explaining that variants
legitimately differ, and collapsing siblings to the best-scoring one, were both tried and
measured: the first changed nothing, the second fixed three questions and broke a fourth.
Neither shipped.

Note what prints under each answer: the source is tagged as a machine reading, and the first one
still carries an extraction warning. That is deliberate — the validator's whitelist is built from
verbatim text only, so it cannot vouch for anything here.

In [12]:
# Answerable only from the scanned signature matrix.
_ = ask("Who approves a comparison sheet over 3M?")

# Two lanes on one page, different thresholds - this is what per-lane chunking buys.
_ = ask("For steel engineering material, at what value does the COO approve?")

# The rejection loop, which OCR would have destroyed entirely.
_ = ask("What happens if the Purchasing Deputy does not approve?")

Q: Who approves a comparison sheet over 3M?
  The comparison sheet over 3M is approved by the CEO. This information is
  specified in the document "CS Signature Matrix.pdf | Application Works
  Approval Flow".

  sources (top-3, route=none):
    [0.945] CS Signature Matrix.pdf  |  Application Works Approval Flow
    [0.661] CS Signature Matrix.pdf  |  Consumable Material
    [0.626] CS Signature Matrix.pdf  |  Engineering Material

  cited forms : none   validator: clean
  latency     : retrieval 10 ms + generation 3.1 s = 3.1 s   (qwen3:14b)



Q: For steel engineering material, at what value does the COO approve?
  For steel engineering material, the COO approves at a value of "Any Value."
  This is specified in the document [CS Signature Matrix.pdf | Engineering
  Material ( Steel )].

  sources (top-3, route=none):
    [1.000] CS Signature Matrix.pdf  |  Engineering Material ( Steel )
    [0.755] CS Signature Matrix.pdf  |  Engineering Material
    [0.625] CS Signature Matrix.pdf  |  Consumable Material

  cited forms : none   validator: clean
  latency     : retrieval 12 ms + generation 3.2 s = 3.2 s   (qwen3:14b)



Q: What happens if the Purchasing Deputy does not approve?
  If the Purchasing Deputy does not approve, the process routes back to the
  Purchasing Project coordinator. This is specified in the document "CS
  Signature Matrix.pdf".

  sources (top-3, route=none):
    [1.000] CS Signature Matrix.pdf  |  Engineering Material
    [0.967] CS Signature Matrix.pdf  |  Consumable Material
    [0.933] CS Signature Matrix.pdf  |  Engineering Material ( Steel )

  cited forms : none   validator: clean
  latency     : retrieval 10 ms + generation 3.2 s = 3.2 s   (qwen3:14b)



---

### What this demo establishes, and what it does not

**Shown:** the pipeline runs end to end on the real PDFs, on this hardware, against a local
`qwen3:14b` — no cloud call, no placeholder numbers. Re-chunking the PDFs reproduces the shipped
`chunks.json` exactly, the validator runs on every answer, latency is measured rather than
estimated, and Arabic questions traverse the same path with the translation hop timed
separately.

**Why `qwen3:14b`:** the 36-question × per-type eval recorded in `model_eval_results.json` —
100% overall, 100% on the `unanswerable` category, zero flagged citations, against a decision
rule fixed before the numbers came in. Read that with one caveat: `gemma4:e4b` scored
*identically* on those 36 questions and was faster on mean latency, so the eval set does not
actually separate the two — it is saturated. The choice of qwen3 rests on the larger model
being the safer default at 500–600 documents, not on a measured win, and a harder eval set
could reopen it. **The harness that produced those numbers is no longer in this repo**, so the
JSON is a record to read, not something this notebook can regenerate.

**Chunking is the part that changed most recently.** `adaptive_chunker.py` replaced literal
`SECTION_HEADERS` matching (§2). On these 9 documents it is a wash by design — retrieval stays
at 100% top-3 and the validator's form whitelist is byte-identical at 67 — so the argument for
it is not accuracy here, it is that the old approach could not survive a document written to a
different template. Two caveats measured while building it are worth carrying: the font-size
signal is nearly inert on this corpus (`numbering` fires 103 times against `font_size`'s 3, so
the layout half is largely untested here), and every threshold in it — 0.85 fuzzy, 0.45 cosine,
1.15 size ratio — is calibrated on these 9 documents.

**The Arabic path is now partly measured, where it used to be only demonstrated.**
`eval_arabic.py` scores retrieval hit on all 36 questions asked in Arabic, against the same gold
documents as the English run, with an untranslated-Arabic control row to show the translation
step is what carries the result. What that does *not* cover is answer quality in Arabic, which
still needs Arabic `must_include` strings written by a person — and the Arabic questions are
machine-generated by the same model that translates them back, so the number is an upper bound
until someone naturalises them. §5.5 has the detail.

**The eval set is narrower than it looks.** All 36 questions are form lookups, deadlines and
role lookups. None asks for a section that is a bare list identified mainly by its heading — so
the 32/32 stayed at 32/32 while *"who are the stakeholders in the vendor selection process"*
returned the wrong section entirely. Saturation hid a whole class of miss, not just the margin
between two models. Any future eval should include one question per section type.

**Not shown:** any behaviour at corpus scale. 9 documents of 500–600.

**Still missing, and it is the next thing to build:** step-level splitting. Both chunkers leave
`step` as `None` on every chunk — the old one had a stage meant to fill it that never matched
anything, the new one does not implement it at all. Harmless at 9 documents where sections are
small; at 500–600 it is what keeps a process step attached to its own form number.

**Scale caveats, unchanged from the handoff:** `PREFIX_HINTS` in `retriever.py` is a hand-written
keyword table — fine for 6 families and 9 documents, but at 500–600 it should be derived from
document titles or routing becomes the thing that silently loses recall. It is also English-only,
which is a second reason Arabic questions are translated before they reach it. FAISS
`IndexFlatIP` stays exact and sub-millisecond to roughly 100k chunks. And the validator's
whitelist only becomes safe as a *blocking* control once the full corpus is indexed.